<a href="https://colab.research.google.com/github/GarimaChaubey/NLP-Natural-Language-Preprocessing-/blob/main/RNN_Beginners_Simple_20Newsgroups_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN for Beginners: Text Classification with PyTorch
## Simple, step-by-step lab using the real 20 Newsgroups dataset

You will learn:
1. Load real text documents.
2. Turn words into integer IDs.
3. Turn IDs into vectors using an embedding layer.
4. Process words in order with an RNN.
5. Train and evaluate a text classifier.

**Main idea:** An RNN reads a sentence one token at a time and carries a hidden state from one token to the next. Run this notebook from top to bottom. The first dataset download requires internet access.


## 1. Import libraries

In [ ]:
import re, random
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2. Load real text data

We use four categories from 20 Newsgroups. The dataset already provides separate training and testing subsets, so we fetch each subset directly.

In [ ]:
CATEGORIES = ["comp.graphics", "rec.sport.baseball", "sci.space", "talk.politics.misc"]
train_data = fetch_20newsgroups(subset="train", categories=CATEGORIES,
    remove=("headers", "footers", "quotes"), shuffle=True, random_state=SEED)
test_data = fetch_20newsgroups(subset="test", categories=CATEGORIES,
    remove=("headers", "footers", "quotes"), shuffle=True, random_state=SEED)
print("Train documents:", len(train_data.data))
print("Test documents:", len(test_data.data))
print("Classes:", train_data.target_names)
print("\nExample text:\n", train_data.data[0][:400])


## 3. Tokenize the text
A token is usually a word or number. We lowercase text and extract simple word/number tokens. This tokenizer is educational, not production-grade.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z]+|[0-9]+", text.lower())

train_tokens = [tokenize(text) for text in train_data.data]
test_tokens = [tokenize(text) for text in test_data.data]
print("Text:", train_data.data[0][:120].replace("\n", " "))
print("Tokens:", train_tokens[0][:20])


## 4. Turn words into integer IDs

Neural networks use numbers rather than strings. We build a vocabulary from training documents only. `<PAD>` is used to fill short sequences; `<UNK>` represents words not found in the vocabulary. Token IDs are lookup numbers, not word meanings.

In [ ]:
PAD_ID, UNK_ID = 0, 1
MAX_VOCAB = 15000
counts = Counter(word for doc in train_tokens for word in doc)
vocab_words = [w for w, c in counts.most_common(MAX_VOCAB - 2)]
word_to_id = {"<PAD>": PAD_ID, "<UNK>": UNK_ID}
for word in vocab_words:
    word_to_id[word] = len(word_to_id)

def encode(words):
    return [word_to_id.get(word, UNK_ID) for word in words]

train_ids = [encode(doc) for doc in train_tokens]
test_ids = [encode(doc) for doc in test_tokens]
print("Vocabulary size:", len(word_to_id))
print("Words:", train_tokens[0][:10])
print("IDs:  ", train_ids[0][:10])


## 5. Make sequences the same length

Documents have different lengths. We keep at most 100 tokens and pad shorter documents with `PAD_ID`. We save each true sequence length so the RNN can ignore padding.

In [ ]:
MAX_LEN = 100
def prepare(ids):
    ids = ids[:MAX_LEN]
    length = max(1, len(ids))
    return ids + [PAD_ID] * (MAX_LEN - len(ids)), length

train_prepared = [prepare(x) for x in train_ids]
test_prepared = [prepare(x) for x in test_ids]
X_train = np.array([x for x, n in train_prepared], dtype=np.int64)
len_train = np.array([n for x, n in train_prepared], dtype=np.int64)
X_test = np.array([x for x, n in test_prepared], dtype=np.int64)
len_test = np.array([n for x, n in test_prepared], dtype=np.int64)
y_train = np.array(train_data.target, dtype=np.int64)
y_test = np.array(test_data.target, dtype=np.int64)
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("First document length:", len_train[0])


### Picture: a sequence of tokens
The IDs below are illustrative. The model processes them in order.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.3))
ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 3)
words = ["NASA", "launched", "a", "rocket"]
for i, word in enumerate(words):
    x = 1.2 + i*2.4
    ax.text(x, 2, word, ha="center", bbox=dict(boxstyle="round,pad=.4", fc="#dbeafe", ec="#2563eb"))
    ax.text(x, 1, f"ID {i+5}", ha="center")
    if i < len(words)-1:
        ax.annotate("", (x+1.5, 2), (x+.6, 2), arrowprops=dict(arrowstyle="->"))
ax.text(5, .25, "Illustrative IDs only", ha="center")
plt.show()


## 6. What is an embedding?

An embedding converts each token ID into a small vector of numbers. For example, a word ID might map to a vector like \([0.2,-0.1,0.7,0.4]\). These are example values only.

The embedding matrix has shape:

\[
E\in\mathbb{R}^{|V|\times d}
\]

Here \(|V|\) is vocabulary size and \(d\) is embedding size. In our model, each token gets a 32-dimensional vector. The model learns these vectors during training.

In [ ]:
EMBED_SIZE = 32
embedding_demo = nn.Embedding(len(word_to_id), EMBED_SIZE, padding_idx=PAD_ID)
demo_ids = torch.tensor(X_train[:2, :5], dtype=torch.long)
demo_vectors = embedding_demo(demo_ids)
print("ID shape:", demo_ids.shape)
print("Embedding shape:", demo_vectors.shape)  # batch, sequence length, embedding size


## 7. How an RNN works

An RNN reads one token at a time. At each step it combines the current word vector with the previous hidden state:

\[
h_t=\tanh(W_xx_t+W_hh_{t-1}+b)
\]

- \(x_t\): current token embedding
- \(h_{t-1}\): information carried from earlier tokens
- \(W_x,W_h,b\): learnable parameters
- \(\tanh\): activation function
- \(h_t\): updated hidden state

The same weights are reused for every token. For classification, we use the final hidden state to predict the document category.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 4)
xs = [1.8, 5, 8.2]
for t, x in enumerate(xs, 1):
    ax.text(x, 2.2, f"RNN cell {t}", ha="center", va="center",
            bbox=dict(boxstyle="round,pad=.5", fc="#dbeafe", ec="#2563eb"))
    ax.text(x, .8, f"word {t}", ha="center",
            bbox=dict(boxstyle="round,pad=.3", fc="#dcfce7", ec="#16a34a"))
    ax.annotate("", (x,1.7), (x,1.15), arrowprops=dict(arrowstyle="->"))
    ax.text(x, 3.25, f"hidden state h{t}", ha="center")
    ax.annotate("", (x,2.7), (x,2.55), arrowprops=dict(arrowstyle="->"))
for i in range(2):
    ax.annotate("", (xs[i+1]-.75,2.2), (xs[i]+.75,2.2), arrowprops=dict(arrowstyle="->", lw=2))
ax.text(5, .2, "Information flows from earlier words to later words", ha="center")
plt.show()


## 8. Prepare mini-batches

Each example contains token IDs, true sequence length, and correct category label. The DataLoader groups examples into batches.

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, X, lengths, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.lengths[idx], self.y[idx]

BATCH_SIZE = 64
train_loader = DataLoader(NewsDataset(X_train, len_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(NewsDataset(X_test, len_test, y_test), batch_size=BATCH_SIZE)
xb, lengths, yb = next(iter(train_loader))
print("Token IDs:", xb.shape, "Labels:", yb.shape)


## 9. Build the RNN classifier

Our model has three parts:
1. `Embedding`: turns IDs into vectors.
2. `RNN`: reads vectors in order.
3. `Linear`: turns the final hidden state into four category scores called logits.

We use packed sequences so the RNN ignores padding positions.

In [ ]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_ID)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, token_ids, lengths):
        embedded = self.embedding(token_ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.rnn(packed)
        final_hidden = hidden[-1]
        return self.fc(final_hidden)

HIDDEN_SIZE = 64
NUM_CLASSES = len(train_data.target_names)
model = SimpleRNNClassifier(len(word_to_id), EMBED_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
print(model)


## 10. Check one forward pass
The model returns four logits per document. Before training, the weights are random, so predictions are not expected to be accurate.

In [ ]:
model.eval()
xb, lengths, yb = next(iter(test_loader))
with torch.no_grad():
    logits = model(xb.to(device), lengths)
print("Input IDs:", xb.shape)
print("Output logits:", logits.shape)
print("First document scores:", logits[0].cpu().numpy())
pred_id = logits[0].argmax().item()
print("Predicted:", train_data.target_names[pred_id])
print("Actual:   ", train_data.target_names[yb[0].item()])


## 11. Train the RNN

Cross-entropy loss measures how well the predicted scores match the correct labels. Adam updates the model weights. `loss.backward()` calculates gradients using backpropagation through time. Gradient clipping helps prevent very large gradients.

For a batch of \(N\) examples, cross-entropy is:

\[
L=-\frac{1}{N}\sum_{i=1}^{N}\log P(y_i\mid x_i)
\]

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def run_epoch(model, loader, training=False):
    model.train() if training else model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    for token_ids, lengths, labels in loader:
        token_ids, lengths, labels = token_ids.to(device), lengths.to(device), labels.to(device)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(token_ids, lengths)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        n = labels.size(0)
        total_loss += loss.item()*n
        total_correct += (logits.argmax(1) == labels).sum().item()
        total += n
    return total_loss/total, total_correct/total


### Train for three epochs first
If this runs comfortably, increase `EPOCHS` to 5 or more.

In [ ]:
EPOCHS = 3
train_losses, test_losses, train_accs, test_accs = [], [], [], []
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = run_epoch(model, train_loader, training=True)
    te_loss, te_acc = run_epoch(model, test_loader, training=False)
    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc); test_accs.append(te_acc)
    print(f"Epoch {epoch}: train loss={tr_loss:.3f}, acc={tr_acc:.3f} | test loss={te_loss:.3f}, acc={te_acc:.3f}")


### Plot training progress

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(train_losses, marker="o", label="Train")
plt.plot(test_losses, marker="o", label="Test")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()
plt.subplot(1,2,2)
plt.plot(train_accs, marker="o", label="Train")
plt.plot(test_accs, marker="o", label="Test")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()
plt.tight_layout(); plt.show()


## 12. Evaluate the model

In [ ]:
model.eval()
all_true, all_pred = [], []
with torch.no_grad():
    for token_ids, lengths, labels in test_loader:
        logits = model(token_ids.to(device), lengths)
        all_pred.extend(logits.argmax(1).cpu().numpy().tolist())
        all_true.extend(labels.numpy().tolist())

print("Test accuracy:", round(accuracy_score(all_true, all_pred), 4))
print(classification_report(all_true, all_pred,
      labels=list(range(NUM_CLASSES)), target_names=train_data.target_names, zero_division=0))


## 13. Why do we need LSTM and GRU?

A basic RNN can struggle to remember information from far earlier tokens. During training, gradients can become very small (**vanishing gradients**) or very large (**exploding gradients**). Gradient clipping helps with exploding gradients.

**LSTM** uses a cell state and gates to decide what to keep, add, and output. A simplified memory update is:

\[
c_t=f_t\odot c_{t-1}+i_t\odot\tilde{c}_t
\]

**GRU** uses update and reset gates, without a separate cell state. LSTM and GRU can often learn longer dependencies more effectively than a basic RNN.

In [ ]:
steps = np.arange(25)
plt.figure(figsize=(7,3.5))
plt.plot(steps, 0.7**steps, label="Repeated factor 0.7")
plt.plot(steps, 1.2**steps, label="Repeated factor 1.2")
plt.yscale("log")
plt.xlabel("Repeated steps"); plt.ylabel("Magnitude (log scale)")
plt.title("Illustration: values can shrink or grow")
plt.legend(); plt.show()


## 14. Beginner exercises

1. Print the first 15 words and IDs of one document.
2. What is the difference between a token ID and an embedding?
3. If batch size is 32, sequence length is 100, and embedding size is 32, what is the embedding tensor shape?
4. Why does an RNN need a hidden state?
5. What does the final linear layer predict?
6. What does `loss.backward()` do?
7. Why can a basic RNN struggle with long sequences?
8. Try changing `nn.RNN` to `nn.GRU`. What needs to change in the model?


## References

- PyTorch RNN: https://pytorch.org/docs/stable/generated/torch.nn.RNN.html
- PyTorch Embedding: https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html
- PyTorch LSTM: https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
- PyTorch GRU: https://pytorch.org/docs/stable/generated/torch.nn.GRU.html
- 20 Newsgroups dataset: https://scikit-learn.org/stable/datasets/real_world.html#the-20-newsgroups-text-dataset

**Tip:** Run all cells from top to bottom. The first data download needs internet. Results vary by hardware and random seed.
